# P3: Oldindan o'qitilgan embeddinglar bilan ishlash
**Mavzu:** so'z vektorlari, kosinus o'xshashlik, so'z analogiyasi, PCA vizualizatsiya
**Kun:** 4-kun amaliyoti — 19-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L3 — Vektor fazo modellari va semantik munosabatlar
**Kapstone modul:** m03 — `PretrainedEmbedder` (`capstone/modules/m03_pretrained_embedder.py`)
**Ma'lumot:** cc_uz_100k.kv (Common Crawl Uzbek Word2Vec, ~240 MB). Offline rejimda
kichik namuna (`d04_checkpoints/uz_mini.vec`) ishlatiladi — 240 MB shart emas.

**Bugungi maqsadlar:**
1. Oldindan o'qitilgan vektorlarni yuklash (Kaggle: gensim `.kv`; offline: `.vec`)
2. Kosinus o'xshashlik orqali eng o'xshash so'zlarni topish (`most_similar`)
3. Vektor arifmetikasi bilan so'z analogiyasini yechish (`b - a + c`)
4. OOV ulushini o'lchash va PCA bilan embeddinglarni 2D da vizualizatsiya qilish
5. m03 `PretrainedEmbedder` ni `capstone/contracts.py` imzolariga muvofiq yozish

| Bo'lim | Vaqt |
|--------|------|
| §1 Muhit tekshiruvi | 3 daq |
| §2 Yaxlit natija (destination first) | 8 daq |
| §3 Tayyor kod bloki — PRIMM | 17 daq |
| §4 Asosiy mavzu — so'nuvchi tayanch | 35 daq |
| §5 Kapstone integratsiya | 13 daq |
| §6 Tadqiqot savoli + yakun | 7 daq |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §1  Muhit tekshiruvi
# ═══════════════════════════════════════════════════════════════
import random, os, sys
from pathlib import Path
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True   # <-- bu qiymatni o'zgartirmang

CHECKPOINT_DIR = Path("d04_checkpoints")
assert CHECKPOINT_DIR.exists(), (
    "d04_checkpoints/ papkasi topilmadi. "
    "Notebookni course/practices/ papkasidan ishga tushiring yoki: git pull."
)

# Kapstone modullari yo'li (m01, m03 import uchun)
REPO_ROOT = Path().resolve().parent.parent
MODULES_DIR = REPO_ROOT / "capstone" / "modules"
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

import sklearn
print(f"Python      : {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy       : {np.__version__}")
try:
    import gensim
    print(f"gensim      : {gensim.__version__} (Kaggle .kv yuklash uchun)")
except ImportError:
    print("gensim      : yo'q (offline rejim — .vec toza numpy bilan o'qiladi)")
print("\n✓ Muhit tayyor.")


---
## §2  Yaxlit Natija — Pirovard Manzil Birinchi! (~8 daqiqa)

Quyida **tugallangan embedding demosi** ishga tushadi: vektorlarni yuklaymiz va
`toshkent` ga eng o'xshash so'zlarni topamiz. Avval natijani ko'ring.


In [ ]:
# Pirovard natija (inline yuklovchi — m03 hali yozilmagan)
from pathlib import Path
import numpy as np

def load_w2v_text(path):
    """word2vec matn formatini gensimsiz o'qiydi -> (words, matritsa, w2i)."""
    words, rows = [], []
    with open(path, encoding="utf-8") as f:
        header = f.readline().split()
        # "n dim" sarlavhasi
        for line in f:
            parts = line.rstrip("\n").split(" ")
            if len(parts) < 2:
                continue
            words.append(parts[0])
            rows.append(np.asarray(parts[1:], dtype=np.float32))
    M = np.vstack(rows)
    w2i = {w: i for i, w in enumerate(words)}
    return words, M, w2i

EMB_FILE = "cc_uz_100k.kv" if not OFFLINE_FALLBACK else "d04_checkpoints/uz_mini.vec"
words, M, w2i = load_w2v_text("d04_checkpoints/uz_mini.vec")   # offline namuna

# L2-normallashtirilgan matritsa (kosinus uchun)
Mn = M / np.linalg.norm(M, axis=1, keepdims=True)

def quick_similar(word, n=5):
    i = w2i[word]
    sims = Mn @ Mn[i]
    order = [j for j in np.argsort(-sims) if j != i][:n]
    return [(words[j], float(sims[j])) for j in order]

print(f"Yuklandi: {len(words)} so'z, {M.shape[1]} o'lchamli vektorlar")
print("\n'toshkent' ga eng o'xshash so'zlar:")
for w, s in quick_similar("toshkent", 5):
    print(f"  {w:14s} {s:.3f}")
print("\n✓ Embedding ishladi! Quyida har qadamni o'rganamiz.")


---
## §3  Tayyor Kod Bloki — PRIMM (~17 daqiqa)

### 3A. Vektorlarni Yuklash

> **Bashorat qiling:**
> Kaggle da haqiqiy `cc_uz_100k.kv` (~240 MB) gensim bilan yuklanadi. Offline esa
> kichik `.vec` namuna. Sizningcha, 100\,000 so'zli model nechta vektor saqlaydi?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# Kaggle (onlayn) da haqiqiy model gensim bilan yuklanadi:
#   from gensim.models import KeyedVectors
#   wv = KeyedVectors.load("cc_uz_100k.kv")        # ~240 MB, 100k so'z
#   words = list(wv.index_to_key); M = wv.vectors
#
# Offline (bu mashg'ulot) da bundled kichik namuna ishlatiladi:
words, M, w2i = load_w2v_text("d04_checkpoints/uz_mini.vec")
Mn = M / np.linalg.norm(M, axis=1, keepdims=True)

print(f"So'zlar soni : {len(words)}")
print(f"Vektor o'lchami: {M.shape[1]}")
print(f"Lug'at namunasi: {words[:8]}")
print(f"'toshkent' vektori (birinchi 5 son): {M[w2i['toshkent']][:5].round(2)}")


> **Tekshiring:**
> 1. `M` matritsasining shakli (`M.shape`) nimaga teng? Qatorlar = so'zlar?
> 2. `Mn` (normallashtirilgan) qatorlarining normasi nechiga teng? (`np.linalg.norm(Mn[0])`)
> 3. Nega kosinus uchun vektorlarni normallashtramiz?

> **O'zgartiring:** `words` ro'yxatidan boshqa so'z tanlab, uning vektorini chop eting.


### 3B. PCA bilan 2D Vizualizatsiya

> **Bashorat qiling:**
> Yuqori o'lchamli vektorlarni PCA bilan 2D ga tushirsak, qaysi so'zlar yaqin
> klasterlarga tushadi? (Shaharlar? Taomlar?)


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(M)
print(f"explained_variance_ratio_: {pca.explained_variance_ratio_.round(3)}")

# Grafik (matplotlib bo'lsa — Kaggle da ko'rinadi)
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 5))
    plt.scatter(xy[:, 0], xy[:, 1], s=12, color="tab:blue")
    for i, w in enumerate(words):
        plt.annotate(w, (xy[i, 0], xy[i, 1]), fontsize=7)
    plt.title("Embeddinglar — PCA 2D")
    plt.tight_layout(); plt.show()
except Exception as ex:
    print(f"(grafik o'tkazib yuborildi: {type(ex).__name__})")
    # Grafiksiz: shaharlar bir-biriga yaqinligini sonli ko'rsatamiz
    ci = w2i["toshkent"]; cj = w2i["samarqand"]; ck = w2i["non"]
    print(f"PCA masofa toshkent-samarqand: {np.linalg.norm(xy[ci]-xy[cj]):.2f}")
    print(f"PCA masofa toshkent-non       : {np.linalg.norm(xy[ci]-xy[ck]):.2f}")


> **Tekshiring:**
> 1. Ikki komponent ma'lumotning qancha qismini saqladi (`explained_variance_ratio_` yig'indisi)?
> 2. Shaharlar (toshkent, samarqand) bir-biriga taomlardan (non) yaqinroqmi?
> 3. 2D rasm — taxminiy. To'liq fazoda yaqinlik kafolatlanadimi? (L3 [J4]-slayd)


In [ ]:
# ─── CHECKPOINT A ─────────────────────────────────────────────────────────
# Yuklangan vektorlar keyingi bo'limlar uchun tayyor (words, M, w2i, Mn).
# Agar yuqoridagi kataklarni o'tkazib yuborgan bo'lsangiz, bu yerdan davom eting:
if "words" not in dir() or "Mn" not in dir():
    words, M, w2i = load_w2v_text("d04_checkpoints/uz_mini.vec")
    Mn = M / np.linalg.norm(M, axis=1, keepdims=True)
print(f"Checkpoint A: {len(words)} so'z tayyor, vektor o'lchami {M.shape[1]}.")


---
## §4  Asosiy Mavzu — So'nuvchi Tayanch (~35 daqiqa)

Uch bosqich: **Namuna** (Men ko'rsataman) → **Birgalikda** (`# === SIZNING KODINGIZ ===`)
→ **Mustaqil** (Siz qilasiz).


### 4A. Namuna: Kosinus O'xshashlik — L3 [I2]-Slayd

L3 ma'ruzasidagi qo'lda misol: $\mathbf{a}=(1,1,1,0)$, $\mathbf{b}=(1,1,0,1)$ uchun
$\cos = 2/3 \approx 0.667$. Buni `cosine_similarity` bilan tekshiramiz.


In [ ]:
# ═══ NAMUNA (Men ko'rsataman): kosinus o'xshashlik ═══
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Ma'ruza L3 [I2]-slayd: "nlp juda qiziq" vs "nlp juda foydali"
# Lug'at: (nlp, juda, qiziq, foydali)
a = np.array([[1, 1, 1, 0]])
b = np.array([[1, 1, 0, 1]])

cos_val = float(cosine_similarity(a, b)[0, 0])
print(f"cosine(a, b) = {cos_val:.4f}   (kutilgan: 2/3 = {2/3:.4f})")

# ─── ASSERT: Ma'ruza L3 [I2]-slayd bilan solishtiring (cos = 2/3 ≈ 0.667) ───
assert abs(cos_val - 0.667) < 1e-3, (
    f"cosine={cos_val:.4f}, kutilgan 0.667. Vektorlarni tekshiring."
)
print("✓ Ma'ruza L3 [I2]-slayd tasdiqlandi: cos = 2/3 ≈ 0.667")


### 4B. Birgalikda: most\_similar — Eng O'xshash So'zlar

Normallashtirilgan matritsa `Mn` da kosinus = skalyar ko'paytma. So'rov so'zining
qatorini butun matritsaga ko'paytirib, eng yuqorilarini topamiz.


In [ ]:
# ═══ BIRGALIKDA: most_similar ═══
def most_similar_local(word, n=5):
    i = w2i[word]
    # === SIZNING KODINGIZ (1 qator) ===
    # Mn[i] vektorini butun Mn matritsasiga skalyar ko'paytiring -> sims
    sims = Mn @ Mn[i]
    order = [j for j in np.argsort(-sims) if j != i][:n]
    return [(words[j], float(sims[j])) for j in order]

natija = most_similar_local("toshkent", 5)
for w, s in natija:
    print(f"  {w:14s} {s:.3f}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
cities = {"samarqand", "buxoro", "andijon", "namangan", "xiva", "nukus"}
top_words = [w for w, _ in natija]
assert len(natija) == 5, "most_similar 5 ta natija qaytarishi kerak."
assert top_words[0] in cities, (
    f"'toshkent' ga eng yaqin so'z o'zbek shahri bo'lishi kutilgan, '{top_words[0]}' chiqdi."
)
assert "moskva" not in top_words[:3], "moskva (boshqa davlat) top-3 da bo'lmasligi kerak."
print(f"✓ most_similar to'g'ri: toshkent ~ {top_words[:3]} (o'zbek shaharlari)")


### 4C. Birgalikda: So'z Analogiyasi — Vektor Arifmetikasi

L3 [I3]-slayd: `uzbekiston : toshkent :: rossiya : ?`. Maqsad vektor
$\mathbf{t} = \text{toshkent} - \text{uzbekiston} + \text{rossiya}$, eng yaqini = javob.


In [ ]:
# ═══ BIRGALIKDA: so'z analogiyasi ═══
def nearest_to_vec(vec, exclude=(), n=1):
    vn = vec / (np.linalg.norm(vec) + 1e-9)
    sims = Mn @ vn
    out = []
    for j in np.argsort(-sims):
        if words[j] in exclude:
            continue
        out.append((words[j], float(sims[j])))
        if len(out) >= n:
            break
    return out

# === SIZNING KODINGIZ (1 qator) ===
# Maqsad vektor: toshkent - uzbekiston + rossiya  (M xom vektorlardan)
t = M[w2i["toshkent"]] - M[w2i["uzbekiston"]] + M[w2i["rossiya"]]

javob = nearest_to_vec(t, exclude={"toshkent", "uzbekiston", "rossiya"}, n=1)
print(f"uzbekiston : toshkent :: rossiya : {javob[0][0]}  (o'xshashlik {javob[0][1]:.3f})")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert t is not None, "Maqsad vektor t None. toshkent - uzbekiston + rossiya ni hisoblang."
assert javob[0][0] == "moskva", (
    f"Analogiya 'moskva' bo'lishi kutilgan, '{javob[0][0]}' chiqdi. "
    "Vektor arifmetikasini tekshiring: b - a + c."
)
print("✓ Analogiya to'g'ri: uzbekiston:toshkent :: rossiya:moskva (poytaxt munosabati)")


### 4D. Birgalikda: OOV Ulushi (oov\_rate)

OOV (out-of-vocabulary) — lug'atda yo'q so'zlar. Ularning ulushi o'zbek tilida
yuqori bo'ladi (L3 [M]-slayd: agglutinatsiya + inglizcha-asosli vektorlar).


In [ ]:
# ═══ BIRGALIKDA: oov_rate ═══
def oov_rate_local(texts):
    """texts: list[list[str]] -> lug'atda yo'q tokenlar ulushi [0,1]."""
    total = oov = 0
    for toks in texts:
        for tok in toks:
            total += 1
            # === SIZNING KODINGIZ (1-2 qator) ===
            # tok lug'atda (w2i) yo'q bo'lsa, oov ni oshiring
            if tok not in w2i:
                oov += 1
    return oov / total if total else 0.0

namuna = [["toshkent", "non", "kompyuter"], ["mashina", "internet", "banan"]]
rate = oov_rate_local(namuna)
print(f"OOV ulushi: {rate:.3f}  ({rate*100:.0f}%)")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
# Lug'atda: toshkent, non, kompyuter, internet (4). OOV: mashina, banan (2). Jami 6.
assert abs(rate - 2/6) < 0.01, (
    f"OOV ulushi {rate:.3f}, kutilgan {2/6:.3f}. Sanashni tekshiring."
)
print(f"✓ oov_rate to'g'ri: 6 tokendan 2 tasi (mashina, banan) OOV -> {rate:.3f}")


### 4E. Mustaqil: O'z So'zlaringiz Bilan Sinab Ko'ring

Scaffold yo'q — yuqorida o'rgangan `most_similar_local` va `nearest_to_vec` dan
foydalaning.


In [ ]:
# ═══ MUSTAQIL (Siz qilasiz) ═══
# 1. Biror so'z tanlang (masalan "non", "kompyuter") va eng o'xshashlarini toping.
# 2. Yangi analogiya yarating (masalan oziq-ovqat yoki texnologiya guruhida).
my_word = "kompyuter"
my_similar = most_similar_local(my_word, 3)
print(f"'{my_word}' ga o'xshash: {[w for w, _ in my_similar]}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert my_similar is not None, "my_similar None. most_similar_local() ni qo'llang."
assert len(my_similar) == 3, "3 ta o'xshash so'z kutilgan."
assert all(isinstance(w, str) for w, _ in my_similar), "Har element (so'z, son) bo'lishi kerak."
print(f"✓ Mustaqil: '{my_word}' ~ {[w for w, _ in my_similar]}")


---
## §5  Loyihaga Ulash — m03 PretrainedEmbedder Yozamiz (~13 daqiqa)

§4 dagi funksiyalarni (most_similar, oov_rate, yuklash) rasmiy kapstone moduliga
o'tkazamiz. Imzolar `capstone/contracts.py` da belgilangan. Modul gensim-ixtiyoriy:
Kaggle da `.kv` (gensim), offline da `.vec` (toza numpy).


In [ ]:
# ═══ §5: m03 PretrainedEmbedder modulini yozamiz ═══
from pathlib import Path
REPO_ROOT = Path().resolve().parent.parent
M03_PATH = REPO_ROOT / 'capstone' / 'modules' / 'm03_pretrained_embedder.py'
M03_PATH.parent.mkdir(parents=True, exist_ok=True)

M03_CODE = r'''"""
capstone/modules/m03_pretrained_embedder.py
PretrainedEmbedder — oldindan o'qitilgan so'z vektorlarini boshqaradi.
Shartnoma: capstone/contracts.py :: PretrainedEmbedder
P3 (4-kun amaliyoti) da qurilgan; m04 (LSH), m08 (RNN init) tomonidan ishlatiladi.

Kaggle: gensim KeyedVectors .kv (cc_uz_100k.kv) yuklanadi.
Offline: word2vec matn formati (.vec) gensimsiz, toza numpy bilan o'qiladi.
"""
from __future__ import annotations

import numpy as np


class PretrainedEmbedder:
    """Oldindan o'qitilgan Word2Vec/.kv embeddinglarini boshqaradi.

    Consumed by: m04 (LSH), m08 (GRU/LSTM pretrained init).
    """

    def __init__(self) -> None:
        self._words: list[str] = []
        self._w2i: dict[str, int] = {}
        self._raw: np.ndarray | None = None   # (n, dim) xom vektorlar
        self._norm: np.ndarray | None = None  # (n, dim) L2-normallashtirilgan (kosinus uchun)
        self._dim: int = 0

    # ─── yuklash ──────────────────────────────────────────────────────────────
    def load(self, path: str) -> None:
        """Gensim KeyedVectors (.kv) yoki word2vec matn (.vec) faylni yuklaydi."""
        if path.endswith(".kv"):
            from gensim.models import KeyedVectors   # faqat Kaggle/onlayn
            kv = KeyedVectors.load(path)
            words = list(kv.index_to_key)
            mat = np.asarray(kv.vectors, dtype=np.float32)
        else:
            words, mat = self._load_text(path)
        self._words = words
        self._w2i = {w: i for i, w in enumerate(words)}
        self._raw = mat.astype(np.float32)
        self._dim = mat.shape[1]
        norms = np.linalg.norm(mat, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self._norm = (mat / norms).astype(np.float32)

    @staticmethod
    def _load_text(path: str) -> tuple[list[str], np.ndarray]:
        words, rows = [], []
        with open(path, encoding="utf-8") as f:
            first = f.readline().split()
            # sarlavha "n dim" bo'lsa o'tkazib yuboramiz, aks holda ma'lumot
            if not (len(first) == 2 and first[0].isdigit() and first[1].isdigit()):
                words.append(first[0])
                rows.append(np.asarray(first[1:], dtype=np.float32))
            for line in f:
                parts = line.rstrip("\n").split(" ")
                if len(parts) < 2:
                    continue
                words.append(parts[0])
                rows.append(np.asarray(parts[1:], dtype=np.float32))
        return words, np.vstack(rows)

    # ─── asosiy metodlar ────────────────────────────────────────────────────────
    def embed(self, word: str) -> np.ndarray:
        """So'z uchun xom vektor; OOV uchun sifr-vektori (shape (dim,) float32)."""
        i = self._w2i.get(word)
        if i is None:
            return np.zeros(self._dim, dtype=np.float32)
        return self._raw[i].copy()

    def most_similar(self, word: str, n: int = 5) -> list[tuple[str, float]]:
        """Kosinus bo'yicha eng o'xshash n ta so'z: [(so'z, o'xshashlik), ...]."""
        i = self._w2i.get(word)
        if i is None:
            return []
        sims = self._norm @ self._norm[i]
        order = np.argsort(-sims)
        out: list[tuple[str, float]] = []
        for j in order:
            if int(j) == i:
                continue
            out.append((self._words[int(j)], float(sims[int(j)])))
            if len(out) >= n:
                break
        return out

    def oov_rate(self, texts: list[list[str]]) -> float:
        """Tokenlar orasida lug'atda yo'q so'zlar ulushi [0,1]."""
        total = oov = 0
        for toks in texts:
            for t in toks:
                total += 1
                if t not in self._w2i:
                    oov += 1
        return oov / total if total else 0.0
'''

M03_PATH.write_text(M03_CODE, encoding='utf-8')
print(f'OK: {M03_PATH} yozildi ({len(M03_CODE)} belgi).')


In [ ]:
# ─── m03 ni import qilib, shartnomaga muvofiqligini tekshiramiz ──────────────
import sys, importlib
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))
import m03_pretrained_embedder
importlib.reload(m03_pretrained_embedder)
from m03_pretrained_embedder import PretrainedEmbedder

emb = PretrainedEmbedder()
emb.load("d04_checkpoints/uz_mini.vec")

# Shartnoma: embed -> np.ndarray, OOV -> sifr-vektori
import numpy as np
v = emb.embed("toshkent")
assert isinstance(v, np.ndarray) and v.shape == (50,), "embed() (50,) ndarray qaytarishi kerak."
assert np.allclose(emb.embed("yoqyoqyoq"), 0.0), "OOV uchun sifr-vektori kutilgan."

# Shartnoma: most_similar -> list[tuple[str, float]]
ms = emb.most_similar("toshkent", 5)
assert len(ms) == 5 and isinstance(ms[0][0], str) and isinstance(ms[0][1], float)
assert ms[0][0] in {"samarqand", "buxoro", "andijon", "namangan", "xiva", "nukus"}

# Shartnoma: oov_rate -> float [0,1]
r = emb.oov_rate([["toshkent", "mashina"]])
assert 0.0 <= r <= 1.0 and abs(r - 0.5) < 1e-9, "oov_rate noto'g'ri."

# Kapstone uzviyligi: m01 bilan tokenize qilib oov_rate
from m01_text_preprocessor import TextPreprocessor
tp = TextPreprocessor()
toks = tp.preprocess("Toshkent va Samarqand go'zal shaharlar")
r2 = emb.oov_rate([toks])
assert 0.0 <= r2 <= 1.0
print("✓ m03 PretrainedEmbedder barcha shartnoma tekshiruvlaridan o'tdi!")
print(f"  most_similar('toshkent')[:3] = {[w for w,_ in ms[:3]]}")
print(f"  m01 tokenlari OOV ulushi      = {r2:.3f}")


### 5C. Git — m03 ni Commit Qiling

```bash
git add capstone/modules/m03_pretrained_embedder.py
git commit -m "day04: capstone - m03 PretrainedEmbedder"
git push origin HEAD
```


In [ ]:
import subprocess
res = subprocess.run(
    ["git", "add", "capstone/modules/m03_pretrained_embedder.py"],
    capture_output=True, text=True, cwd=str(REPO_ROOT))
print("git add:", "OK" if res.returncode == 0 else res.stderr.strip())
st = subprocess.run(["git", "diff", "--cached", "--name-only"],
                    capture_output=True, text=True, cwd=str(REPO_ROOT))
if st.stdout.strip():
    cm = subprocess.run(
        ["git", "commit", "-m", "day04: capstone - m03 PretrainedEmbedder"],
        capture_output=True, text=True, cwd=str(REPO_ROOT))
    print(cm.stdout.strip() or "Commit bajarildi.")
else:
    print("Commit qilish uchun yangi o'zgarish yo'q.")


---
## §6  Tadqiqot Savoli + Yakun (~7 daqiqa)

**Savol:** O'zbek matnlarida OOV ulushi nega yuqori? Apostrof variantlari
(`o'` / `oʻ` / `o`) bir xil so'zni turli token qiladi — bu embeddingga qanday ta'sir qiladi?


In [ ]:
# Mini tadqiqot: apostrof varianti OOV ga qanday ta'sir qiladi?
variantlar = ["to'g'ri", "tog'ri", "togri"]   # ASCII ', U+2019 yo'q, apostrofsiz
for w in variantlar:
    bor = w in w2i
    print(f"  {w:10s} lug'atda: {'BOR' if bor else 'YO'+chr(39)+'Q (OOV)'}")

# Agglutinatsiya: bir o'zak, ko'p shakl
shakllar = [["toshkent", "toshkentda", "toshkentdan", "toshkentga"]]
print(f"\nAgglutinativ shakllar OOV ulushi: {oov_rate_local(shakllar):.3f}")
print("Mulohaza: faqat 'toshkent' lug'atda; qo'shimchali shakllar OOV.")
print("Yechim: m01 bilan normallashtirish/stemming -> OOV kamayadi (P1 dan eslang).")


---
## Yakun

**Bugun nimalar qildik:**
- ✓ Oldindan o'qitilgan vektorlarni yukladik (Kaggle: gensim `.kv`; offline: `.vec`)
- ✓ Kosinus o'xshashlikni tasdiqladik — L3 [I2]-slayd: cos = 2/3 ≈ 0.667
- ✓ `most_similar` bilan eng o'xshash so'zlarni topdik (toshkent ~ o'zbek shaharlari)
- ✓ Vektor arifmetikasi bilan analogiyani yechdik (uzbekiston:toshkent :: rossiya:moskva)
- ✓ OOV ulushini o'lchadik va PCA bilan vizualizatsiya qildik
- ✓ m03 `PretrainedEmbedder` ni capstone moduliga yozdik

**Ertaga (P4 — imlo tuzatish va LSH qidiruv):**
m01 -> edit_distance (DP) -> noisy channel -> MinHash LSH -> m04 SpellLSHRetriever
Birinchi assert: `edit_distance("qo'l","ko'l")==1` (L4 [I3]-slayd)

---

> **Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima?
> (Quyidagi katakka yozing — keyingi darsda muhokama qilamiz.)
